In [2]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)

# Change the Drive letter to "F" or "D" depending on the mounted drive in the PC

#======================================================================================================================
# Select this path  for training data
#FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Mirage")

# Select this path  for testing data
FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Mirage")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 54 matches to update. Processing...

Successfully updated: 241e1655-c5da-47a1-8691-63e0965c2eea
Successfully updated: 844bfe9e-1d84-480b-8c49-0db28c636421
Successfully updated: c98aa6bf-ccfe-4645-8137-00d0888fe60d
Successfully updated: 31b88928-c71f-43ab-a587-13157b3a70f6
Successfully updated: 6d2daac1-e77b-4e66-a932-28a79b3b6880
Successfully updated: 67a18782-7bab-44ae-9cef-0c8f64e6fcc8
Successfully updated: 064431dd-f86f-4423-af82-d0c37eeb4472
Successfully updated: 3abe2c18-2c09-4f0b-97c8-87228f0cd0b2
Successfully updated: b7962429-c173-4949-941a-af982d5d9cb0
Successfully updated: dff3938c-2600-47e4-b7dd-6c8860d8ef9b
Successfully updated: ec826a68-8d99-4b01-a746-b0b24432b0e9
Successfully updated: 02c7915b-4417-4c36-8edb-9a3d94bf92b5
Successfully updated: 084b7fdb-e7c1-4c87-af34-9ff23ff7f969
Successfully updated: 3820e01d-761f-4b53-b44b-4496bdd8078b
Successfully updated: 5275a871-7152-4686-bd91-060c94219faf
Successfully updated: a9057dec-194b-442b-92fe-b82aa0237187
Successfully 